# 01 — Get the data

Stage 1 of 4. Downloads (or loads) the raw data and saves an untouched copy to `data/raw/`.

**Source:** Horse Racing dataset by hwaitt (Kaggle), https://www.kaggle.com/datasets/hwaitt/horse-racing, downloaded [DATE].
**Licence:** CC BY-NC 4.0, https://creativecommons.org/licenses/by-nc/4.0/ (Requirement 4.15). Raw files are not committed.


**Input:** none (needs Kaggle credentials in `~/.kaggle/kaggle.json`, see README)
**Output:** `data/raw/races_<year>.csv` and `data/raw/horses_<year>.csv`, 2010-2020

Only the races and horses files are used. `forward.csv` is not downloaded into `data/raw/`: it covers only Sep-Dec 2020 and has no outcomes.

In [6]:
import sys
from pathlib import Path

# Make code/helpers.py importable whether the notebook is run from code/ or the repo root.
sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "helpers.py").exists() else Path.cwd() / "code"))

import pandas as pd
import shutil
import kagglehub

import helpers
from helpers import DATA_RAW

helpers.set_seed()
helpers.ensure_dirs()

YEARS = range(2010, 2021)  # 2010-2020 inclusive

In [7]:
# Drafted with Claude: download stage for the Kaggle horse-racing data.
# Credentials, if the download needs them, come from ~/.kaggle/kaggle.json (never put a key in this notebook).
# Pinned to dataset version 3 so reruns get the same data.
expected = [f"{prefix}_{y}.csv" for prefix in ("races", "horses") for y in YEARS]
missing = [n for n in expected if not (DATA_RAW / n).exists()]

if missing:
    cache_dir = Path(kagglehub.dataset_download("hwaitt/horse-racing/versions/3"))
    all_files = sorted(cache_dir.rglob("*.csv"))
    print(len(all_files), "csv files found; first 10:")
    print(*[p.relative_to(cache_dir) for p in all_files[:10]], sep="\n")

    kept = []
    for p in all_files:
        name = p.name.lower()
        if name.startswith(("races", "horses")) and any(str(y) in name for y in YEARS):
            shutil.copy2(p, DATA_RAW / p.name)   # raw copy, unchanged
            kept.append(p.name)
    print(f"Kept {len(kept)} files (expected {len(expected)})")
    assert len(kept) == len(expected), "Unexpected file count: check the names printed above"
else:
    print("All files already in data/raw/, skipping download")

All files already in data/raw/, skipping download


In [8]:
for prefix in ("races", "horses"):
    f = sorted(DATA_RAW.glob(f"{prefix}*"))[0]
    df = pd.read_csv(f)
    print(f.name, df.shape)
    display(df.head(3))

races_2010.csv (15215, 18)


,rid,course,time,date,title,rclass,band,ages,distance,condition,hurdles,prizes,winningTime,prize,metric,countryCode,ncond,class
0,272760,Cheltenham,03:50,10/01/01,"EBF ""High Sheriff of Gloucestershire's"" ""Junio...",Class 1,NaN,4yo,1m6f,Soft,NaN,"[10115.0, 3787.6, 1892.1, 945.2]",222.70,16740.0,2815.0,GB,5,1
1,280996,Cheltenham,01:40,10/01/01,Dipper Novices' Chase Grade 2,Class 1,NaN,5yo+,2m5f,Soft,17 fences,"[18243.2, 6844.8]",357.30,25088.0,4223.0,GB,5,1
2,286408,Southwell (AW),01:35,10/01/01,Betdaq The Betting Exchange Handicap,Class 5,0-75,4yo+,1m6f,Standard To Slow,NaN,"[2331.36, 693.72, 346.68, 173.16]",195.35,3545.0,2815.0,GB,13,5


horses_2010.csv (157181, 27)


,rid,horseName,age,saddle,decimalPrice,isFav,trainerName,jockeyName,position,positionL,...,TR,OR,father,mother,gfather,runners,margin,weight,res_win,res_place
0,272760,Tetlami,4.0,3.0,0.200000,1,Nicky Henderson,Barry Geraghty,1,NaN,...,NaN,NaN,Daylami,Tetou,Peintre Celebre,15,1.208562,70,1.0,1
1,272760,Romulus D'Artaix,4.0,14.0,0.142857,0,Alan King,Robert Thornton,2,4,...,NaN,NaN,Sassanian,Kadisha,Always Fair,15,1.208562,68,0.0,1
2,272760,Marleno,4.0,9.0,0.024390,0,Milton Harris,Warren Marston,3,nk,...,NaN,NaN,Lecroix,Mondalita,Alkalde,15,1.208562,68,0.0,1


In [9]:
rows = {}
for f in sorted(DATA_RAW.glob("*.csv")):
    rows[f.name] = sum(1 for _ in open(f, "rb")) - 1   # data rows, minus header
summary = pd.Series(rows, name="rows")
print(summary.to_string())
print("Total rows:", summary.sum())

horses_2010.csv    157181
horses_2011.csv    167448
horses_2012.csv    163080
horses_2013.csv    164852
horses_2014.csv    160577
horses_2015.csv    156753
horses_2016.csv    157328
horses_2017.csv    166865
horses_2018.csv    171195
horses_2019.csv    171849
horses_2020.csv    149513
races_2010.csv      15215
races_2011.csv      16571
races_2012.csv      16343
races_2013.csv      16988
races_2014.csv      17042
races_2015.csv      16597
races_2016.csv      16252
races_2017.csv      17126
races_2018.csv      17553
races_2019.csv      17307
races_2020.csv      14794
Total rows: 1968429
